<a href="https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ifeoluwa-Analytics/FlyRank-AI---ML-Track/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My Lane Task Type: Supervised Classification & Probability-Based Ranking.

Why: Predicting content decay requires estimating the probability that a visible, mature content item will experience sustained traffic loss (trend_direction == "down"). The output is sorted into a prioritized Top-50 review queue so editorial teams review the highest-risk pages first.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Variable: Binary flag is_declining_label = (trend_direction == "down")
.

Source of Label: In the starter dataset, it is an observed categorical trend bucket representing sustained loss of traffic/impressions over prior windows.

Prediction Discipline: Features are calculated strictly from past measurements (impressions_90d, sessions_90d, average_position, content_age_days), excluding any future-window metrics or existing product rule flags (like health_score) to avoid circular learning

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric/Metric to Defend: Precision@50 (and Average Precision).

What Means 'Good': Precision@50 ≥ 0.70. A standard heuristic baseline rule achieves 0.240 Precision@50 (getting ~12 out of 50 correct)
. A successful ML model should achieve ~0.740 Precision@50 (getting ~37 out of 50 correct), beating the baseline score and thus nearly tripling the efficiency of the human editorial review queue.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis (Grain): One row = One unique content item (content_id / content_hash_id)

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np

# Loading data & applying population filters (unit of analysis)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df_filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df_filtered["is_declining"] = df_filtered["trend_direction"] == "down"

# Displaying the real dataframe slice
print(f"Unit of Analysis Grain: One row = One Content Item")
print(f"Dataframe Shape: {df_filtered.shape[0]:,} rows x {df_filtered.shape[1]} columns")
df_filtered[["content_id", "impressions_90d", "sessions_90d", "avg_position", "content_age_days", "is_declining"]].head()

Unit of Analysis Grain: One row = One Content Item
Dataframe Shape: 30,000 rows x 45 columns


,content_id,impressions_90d,sessions_90d,avg_position,content_age_days,is_declining
0,content_304f48230142,3803,17,10.6,187,True
1,content_a1fb4e703a9e,15320,9,20.3,445,True
2,content_9aa793d4d895,12581,11,36.5,141,True
3,content_331d6c4de07b,11751,78,6.2,463,False
4,content_d99b7a2d90ca,19140,145,44.0,263,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Fixed if/else heuristic rules (e.g., "flag if age > 180 days AND impressions < 500") fail because content decay is a multi-dimensional interaction, and fixed  single-variable rules fail on multi-dimensional trade-offs.
Displaying the dataframe slice in the section 4 above makes it visually obvious that single metrics (like age or position alone) do not linearly determine whether a page declines. We need a Machine Learning model (e.g Random Forest) that can evaluate age, position, impressions, and engagement all at once to accurately predict that is_declining column.

From the analysis above, Row 0 (content_304f48230142): Age is 187 days, Position is 10.6, Sessions = 17 $\rightarrow$ is_declining = True.Row 3 (content_331d6c4de07b): Age is 463 days (which is much older), Position is 6.2 (top of page 1), Sessions = 78 $\rightarrow$ is_declining = False.
If we used a simple heuristic rule like "flag any article older than 180 days," we would wrongly flag Row 3 (a healthy page bringing in steady traffic).

The ML model however looks at age, average position, impressions, and engagement together to recognize that Row 3 is performing well despite its age, while Row 0 actually needs attention.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.